# Demo: tái hiện V-SQL, cải tiến V-SQL+, đánh giá so với DAIL-SQL

BIRD mini-dev 2025-07-04 (500 câu, 11 CSDL). So sánh **DAIL-SQL** (baseline trong Bảng 3 của paper), **V-SQL** (tái hiện; đưa hết
view / LLM chọn view) và **V-SQL+** (cải tiến, bản bỏ bầu chọn nhiều mẫu).

| | Ô | Nội dung | Thời gian |
|---|---|---|---|
| **Chuẩn bị** | A–D | Model, dữ liệu; dựng view, chỉ mục giá trị, tập ví dụ DAIL-SQL; chạy sẵn 10 câu demo | Máy: khoảng 20 giây. Colab: khoảng 2–3 phút |
| **Demo** (khoảng 3 phút) | 1–2 | Tái hiện V-SQL, cải tiến V-SQL+ | Tức thì |
| | 3–5 | Chạy các phương pháp trên câu demo, xem luồng gọi LLM, số lần gọi, token | Khoảng 1 giây, chạy lại thoải mái |
| | 6 | Kết quả trên đủ 500 câu | Tức thì |

**Chi phí:** với `gpt-4o-mini`, câu trả lời của 10 câu demo đã có sẵn trong cache nên không gọi API, không tốn tiền.

Cách chạy và lời thoại quay video: `docs/HUONG_DAN_DEMO_VIDEO.md`.

## Chuẩn bị (chạy ô A → D trước khi trình bày)

In [ ]:
#@title A. Model
#@markdown Tên model OpenAI, hoặc model Ollama có dấu `:` (vd `qwen2.5-coder:7b`, chạy trên Colab T4).
#@markdown `gpt-4o-mini` dùng cache có sẵn, không cần API key. Model khác cần `OPENAI_API_KEY` và gọi API thật.
MODEL = "gpt-4o-mini"  #@param ["gpt-4o-mini", "gpt-4.1-mini", "gpt-4o", "qwen2.5-coder:7b"] {allow-input: true}
MODEL = MODEL.strip()
BACKEND = "ollama" if ":" in MODEL else "openai"
print(f"Model {MODEL} ({BACKEND})")

In [ ]:
#@title B. Mã nguồn, thư viện, dữ liệu (Colab: khoảng 1 phút)
import os, sys, json, subprocess, hashlib, urllib.request, time
from pathlib import Path
TREN_COLAB = "google.colab" in sys.modules


def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


T0 = time.time()
# Gói dữ liệu demo (266 MB): 6 CSDL của 10 câu demo, file câu hỏi, tập train DAIL-SQL đã xử lý sẵn, cache câu trả lời gpt-4o-mini.
# Tải nhanh hơn nhiều so với minidev.zip (1,4 GB, máy chủ ở Bắc Kinh) và không phải tính lại tập train DAIL-SQL (~8 phút).
PACK_URL = "https://github.com/chloenguyen167/vsqlplus/releases/download/demo-data-v1/vsql_demo_pack.zip"
PACK = Path("/content/vsql_demo_pack")
if TREN_COLAB:
    REPO = Path("/content/vsqlplus")
    if REPO.is_dir():
        sh(f"git -C {REPO} pull -q")
    else:
        sh(f"git clone -q --depth 1 https://github.com/chloenguyen167/vsqlplus.git {REPO}")
    sh('pip install -q "sql_metadata<3"')
    try:                                      # Colab thường có sẵn sentence-transformers và spaCy: chỉ cài khi thiếu
        import sentence_transformers, spacy; spacy.load("en_core_web_sm")
    except Exception:
        sh("pip install -q sentence-transformers spacy")
        sh("python -m spacy download en_core_web_sm -q")
    if not (PACK / "MINIDEV/dev_databases").is_dir():
        try:
            sh(f"wget -q -O /content/vsql_demo_pack.zip {PACK_URL}")
            sh("unzip -q -o /content/vsql_demo_pack.zip -d /content && rm /content/vsql_demo_pack.zip")
        except subprocess.CalledProcessError:
            print("Không tải được gói dữ liệu demo -> tải minidev.zip đầy đủ (chậm hơn nhiều)")
    if (PACK / "MINIDEV/dev_databases").is_dir():
        DATA, TRAIN_DIR = PACK / "MINIDEV", PACK / "bird_train"
    else:
        DATA = Path("/content/data/minidev/MINIDEV")
        if not (DATA / "dev_databases").is_dir():
            os.makedirs("/content/data", exist_ok=True)
            sh("wget -q -O /content/data/minidev.zip https://bird-bench.oss-cn-beijing.aliyuncs.com/minidev.zip")
            sh('unzip -q -o /content/data/minidev.zip "minidev/MINIDEV/*" -d /content/data')
        TRAIN_DIR = Path("/content/data/bird_train")
    OUT = Path("/content/demo_vsql")
else:
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_compare.py").exists())
    DATA = REPO / "minidev/MINIDEV"
    TRAIN_DIR = REPO / "minidev/bird_train"
    OUT = REPO / "results_new/demo_vsql"
os.chdir(REPO)
sys.path.insert(0, str(REPO))

DATA_JSON = DATA / "mini_dev_sqlite_2025-07-04.json"
if not DATA_JSON.exists():
    urllib.request.urlretrieve("https://huggingface.co/datasets/birdsql/bird_mini_dev/resolve/"
                               "f65faf4ae3b638c1fa6df1d3370c8d92c8366301/data/mini_dev_sqlite-00000-of-00001.json", DATA_JSON)
assert hashlib.md5(DATA_JSON.read_bytes()).hexdigest() == "595b6b014a528f4386d2e16f74d14b40", "Sai file câu hỏi 2025-07-04"

try:                                          # DAIL-SQL che tên schema bằng spaCy; thiếu model thì chọn ví dụ khác lần chạy 500 câu
    import spacy; spacy.load("en_core_web_sm")
except Exception:
    print("CẢNH BÁO: thiếu spaCy en_core_web_sm -> DAIL-SQL chọn ví dụ khác lần chạy 500 câu (không dùng được cache). "
          "Trên máy: mở Jupyter bằng .venv/bin/jupyter.")
from vsql.baselines import fetch_bird_train
if not (TRAIN_DIR / "train.json").exists():
    fetch_bird_train(str(TRAIN_DIR))          # chỉ lấy train.json + train_tables.json trong train.zip (HTTP Range)

OUT.mkdir(parents=True, exist_ok=True)
if MODEL == "gpt-4o-mini" and not (OUT / "llm_cache.sqlite").exists():
    import sqlite3                            # cache 500 câu nếu máy có, không thì cache 10 câu demo trong gói dữ liệu
    nguon = next(p for p in [REPO / "results_new/cmp_gpt-4o-mini_d2025/llm_cache.sqlite",
                             PACK / "llm_cache_gpt-4o-mini_demo.sqlite"] if p.exists())
    with sqlite3.connect(nguon) as a, sqlite3.connect(OUT / "llm_cache.sqlite") as b:
        a.backup(b)                           # chép bản sao: demo không ghi vào kết quả 500 câu
    print("Dùng cache câu trả lời có sẵn:", nguon.name)

if BACKEND == "openai":
    from vsql.llm import find_api_key
    if not find_api_key() and MODEL != "gpt-4o-mini":
        print("CHƯA CÓ OPENAI_API_KEY: model này cần key (Colab: thêm secret OPENAI_API_KEY, bật quyền truy cập sổ tay).")
elif TREN_COLAB:                              # Ollama trên Colab T4
    sh("apt-get -qq install -y zstd pciutils > /dev/null")
    sh("curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1")
    if not globals().get("ollama_proc") or ollama_proc.poll() is not None:
        ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT,
                                       env=dict(os.environ, OLLAMA_KEEP_ALIVE="-1", OLLAMA_HOST="127.0.0.1:11434"))
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2); break
        except Exception:
            time.sleep(1)
    sh(f"ollama pull {MODEL}")
print(f"Sẵn sàng sau {time.time() - T0:.0f} giây. Repo: {REPO} | dữ liệu: {DATA}")

In [ ]:
#@title C. Chuẩn bị các phần chạy lâu: view, chỉ mục giá trị, tập ví dụ DAIL-SQL, chạy sẵn 10 câu demo
import copy, html
from collections import Counter, defaultdict
import run_compare as RC
from vsql.methods import DbCtx, load_tables_json, run_item, score
from vsql.baselines import dail_index
from vsql.db import run_sql
from vsql.llm import OllamaLLM, OpenAICompatLLM
from IPython.display import HTML, Markdown, display

t0 = time.time()
argv = ["--data", str(DATA_JSON), "--db-root", str(DATA / "dev_databases"), "--tables-json", str(DATA / "dev_tables.json"),
        "--backend", BACKEND, "--model", MODEL, "--out", str(OUT), "--no-embed",
        "--dail-train", str(TRAIN_DIR / "train.json"), "--dail-train-tables", str(TRAIN_DIR / "train_tables.json"),
        "--dail-cache", str(TRAIN_DIR)]
ARGS = RC.parse_args(argv)
data = RC.load_data(ARGS)
gold_by_db = defaultdict(list)
for it in data:
    gold_by_db[it["db_id"]].append(it["SQL"])     # view của V-SQL nhìn JOIN của mọi SQL đáp án trong CSDL (như paper, mục 6)

# 10 câu demo cố định, sắp sao cho từ 3 câu trở lên có đủ 3 độ khó; 10 câu = 3 dễ / 5 trung bình / 2 khó như bộ dữ liệu.
# Ghi chú là kết quả của gpt-4o-mini trong lần chạy 500 câu.
DS_CAU = [
    (259, "V-SQL đúng, DAIL-SQL sai", "View v_superhero (đúng Hình 3) có sẵn alignment, publisher_name nên không cần JOIN; "
          "DAIL-SQL đoán cột superhero.alignment (không tồn tại)"),
    (330, "V-SQL+ sửa được lỗi của V-SQL", "Hình 4 không nhắc SQLite nên V-SQL chép hàm DIVIDE, YEAR từ evidence; prompt V-SQL+ ghi rõ SQLite"),
    (261, "V-SQL+ sửa được lỗi của V-SQL", "D′ của V-SQL ẩn bảng tra cứu publisher nên không lấy được publisher.id; V-SQL+ vẫn giữ bảng này"),
    (380, "V-SQL đúng, DAIL-SQL sai", "DAIL-SQL nối sai khoá cards.uuid = setCode; V-SQL nối đúng theo setCode"),
    (427, "V-SQL đúng, DAIL-SQL sai", "DAIL-SQL lấy atom_id từ nhầm bảng; V-SQL dùng v_connected và v_bond"),
    (266, "Chỉ V-SQL+ đúng", "Liên kết giá trị: CSDL ghi 'Human' (viết hoa); DAIL-SQL và V-SQL viết 'human' nên ra rỗng"),
    (421, "V-SQL đúng, DAIL-SQL sai", "DAIL-SQL JOIN thừa bảng molecule và lấy element từ nhầm bảng; v_atom không cần JOIN"),
    (265, "Chỉ DAIL-SQL đúng", "Giai đoạn 1 của V-SQL và V-SQL+ đúng, nhưng Hình 5 viết lại eye_colour = 'No Colour' thành "
          "eye_colour_id IS NULL"),
    (50, "Chỉ DAIL-SQL đúng", "Phải viết 'Women''s Soccer'; DAIL-SQL học từ ví dụ, V-SQL và V-SQL+ (zero-shot) bị lỗi cú pháp"),
    (85, "V-SQL+ sai", "V-SQL+ đếm trên v_Examination (mỗi lần khám một dòng) thay vì bảng Patient"),
]
GHI_CHU = {u: (a, b) for u, a, b in DS_CAU}
TAT_CA = ["dailsql", "vsql_all", "vsql_llm", "vsqlplus"]
TEN = {"dailsql": "DAIL-SQL", "vsql_all": "V-SQL (đưa hết view)", "vsql_llm": "V-SQL (LLM chọn view)",
       "vsqlplus": "V-SQL+ (bỏ bầu chọn)"}
METHOD = {"dailsql": "dailsql", "vsql_all": "vsql", "vsql_llm": "vsql", "vsqlplus": "vsqlplus_novote"}

# 1) View V-SQL (theo JOIN đáp án), view tự động V-SQL+, chỉ mục giá trị CSDL (liên kết giá trị của V-SQL+)
tables_json = load_tables_json(str(DATA / "dev_tables.json"))
CTX = {}
for db in sorted({data[u]["db_id"] for u, _, _ in DS_CAU} | {"superhero"}):
    c = CTX[db] = DbCtx(db, DATA / "dev_databases" / db, gold_by_db[db], ARGS, tables_json)
    c.oracle, c.auto, c.vindex
print(f"1) View và chỉ mục giá trị của {len(CTX)} CSDL: {time.time() - t0:.0f} giây", flush=True)

# 2) Tập ví dụ DAIL-SQL: nạp mô hình embedding all-mpnet-base-v2 + 9.428 câu train đã xử lý sẵn
t1 = time.time()
dail_index(ARGS)
print(f"2) Tập ví dụ DAIL-SQL: {time.time() - t1:.0f} giây", flush=True)

# 3) LLM (có ghi nhật ký từng lần gọi) và chạy sẵn 10 câu × 4 phương pháp, để lúc demo mọi thứ đã nằm trong bộ nhớ và cache
if BACKEND == "openai":
    LLM = OpenAICompatLLM(MODEL, ARGS.base_url, None, str(OUT / "llm_cache.sqlite"), ARGS.num_predict)
else:
    LLM = OllamaLLM(MODEL, ARGS.host, str(OUT / "llm_cache.sqlite"), ARGS.num_ctx, ARGS.num_predict, ARGS.think)


class GhiNhatKy:
    """Bọc client LLM: ghi lại mọi lần gọi (giai đoạn, prompt, câu trả lời, token, lấy từ cache hay gọi thật)."""

    def __init__(self, llm):
        self.llm, self.log = llm, []

    def chat(self, prompt, meta=None, **opts):
        tu_cache = self.llm.cached(prompt, **opts) is not None
        r = self.llm.chat(prompt, meta, **opts)
        self.log.append(dict(stage=(meta or {}).get("stage", "?"), prompt=prompt, opts=opts, text=r["text"],
                             pt=r["prompt_tokens"], ct=r["completion_tokens"], tu_cache=tu_cache))
        return r


def args_cho(k):
    a = copy.copy(ARGS)
    a.vsql_select = {"vsql_all": "all", "vsql_llm": "llm"}.get(k, ARGS.vsql_select)
    return a


GOLD = {u: run_sql(CTX[data[u]["db_id"]].path, data[u]["SQL"], ARGS.timeout * 10)[0::2] for u, _, _ in DS_CAU}


def chay(it, k):
    """Chạy một phương pháp trên một câu, chấm bằng cách so kết quả với SQL đáp án."""
    g = GhiNhatKy(LLM)
    rec = run_item(METHOD[k], it, CTX[it["db_id"]], g, args_cho(k))
    return dict(rec=score(rec, *GOLD[it["uid"]], CTX[it["db_id"]].path, ARGS.timeout), log=g.log)


t2 = time.time()
goi_that = 0
for u, _, _ in DS_CAU:
    for k in TAT_CA:
        goi_that += sum(not x["tu_cache"] for x in chay(data[u], k)["log"])
print(f"3) Chạy sẵn 10 câu × 4 phương pháp: {time.time() - t2:.0f} giây, {goi_that} lần gọi API thật (lần sau lấy từ cache)")
print(f"\nĐã chuẩn bị xong sau {time.time() - t0:.0f} giây (tính từ ô B: {time.time() - T0:.0f} giây). "
      "Các ô demo 1–6 giờ chạy trong khoảng 1 giây.")

In [ ]:
#@title D. Hàm hiển thị (bảng, chi tiết từng câu, biểu đồ)
GIAI_DOAN = {
    "dail_pre": "Vòng 1: 7 ví dụ chọn theo câu hỏi → SQL nháp",
    "dail_final": "Vòng 2: 7 ví dụ chọn theo khung SQL nháp → SQL cuối",
    "vsql_sel": "Chọn view/bảng của D′ (LLM; phần tự thêm)",
    "vsql_s1": "Giai đoạn 1 (Hình 4): dummy SQL trên view",
    "vsql_s2": "Giai đoạn 2 (Hình 5): viết lại về bảng gốc",
    "plus_final": "Giai đoạn 1 (prompt V-SQL+): dummy SQL trên view",
    "plus_fb0": "Sửa lỗi lần 1", "plus_fb1": "Sửa lỗi lần 2",
    "plus_s2": "Giai đoạn 2 (Hình 5): viết lại về bảng gốc",
}
NHOM = {"vsql_sel": "Chọn view (LLM)", "dail_pre": "Giai đoạn 1 / vòng 1", "vsql_s1": "Giai đoạn 1 / vòng 1",
        "plus_final": "Giai đoạn 1 / vòng 1", "plus_fb0": "Sửa lỗi", "plus_fb1": "Sửa lỗi",
        "dail_final": "Giai đoạn 2 / vòng 2", "vsql_s2": "Giai đoạn 2 / vòng 2", "plus_s2": "Giai đoạn 2 / vòng 2"}
THU_TU = ["Chọn view (LLM)", "Giai đoạn 1 / vòng 1", "Sửa lỗi", "Giai đoạn 2 / vòng 2"]
MAU = dict(zip(THU_TU, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))   # 4 màu đầu bảng phân loại, đã kiểm tra mù màu


def pre(s, cat=6000):
    s = s or ""
    if cat and len(s) > cat:
        s = s[:cat] + f"\n\n… (đã cắt, còn {len(s) - cat:,} ký tự)"
    return ("<pre style='white-space:pre-wrap;font-size:12px;max-height:420px;overflow:auto;"
            "background:#f6f6f4;padding:8px;border-radius:4px'>" + html.escape(s) + "</pre>")


def sql_dap_an(s):
    """Khung nổi bật cho SQL đáp án: nền vàng nhạt, viền trái đậm, chữ đen (đọc được cả khi Colab dùng giao diện tối)."""
    return ("<div style='margin:10px 0;border-left:6px solid #eda100;background:#fff4d6;color:#1f1f1f;padding:8px 12px;"
            "border-radius:4px'><div style='font-size:15px;font-weight:700;margin-bottom:4px'>🎯 SQL đáp án (gold)</div>"
            "<pre style='white-space:pre-wrap;font-size:13px;margin:0;background:transparent;color:#1f1f1f;"
            "font-weight:600'>" + html.escape(s or "") + "</pre></div>")


def bang_ket_qua(cau, cau_hinh, kq):
    dau = lambda r: "✅" if r["correct"] else "❌"
    rows = ["<tr><th>uid</th><th>CSDL</th><th>Độ khó</th><th>Điểm minh hoạ</th>"
            + "".join(f"<th>{TEN[k]}</th>" for k in cau_hinh) + "</tr>"]
    for it in cau:
        rows.append(f"<tr><td>{it['uid']}</td><td>{it['db_id']}</td><td>{it['difficulty']}</td>"
                    f"<td>{html.escape(GHI_CHU[it['uid']][0])}</td>"
                    + "".join(f"<td style='text-align:center'>{dau(kq[(it['uid'], k)]['rec'])}</td>" for k in cau_hinh) + "</tr>")
    rows.append("<tr><td colspan=4><b>Số câu đúng</b></td>"
                + "".join(f"<td style='text-align:center'><b>{sum(kq[(it['uid'], k)]['rec']['correct'] for it in cau)}"
                          f"/{len(cau)}</b></td>" for k in cau_hinh) + "</tr>")
    return HTML("<table>" + "".join(rows) + "</table>")


def chi_tiet(it, cau_hinh, kq):
    out = [f"<h3>uid {it['uid']} · {it['db_id']} · {it['difficulty']} — {html.escape(GHI_CHU[it['uid']][0])}</h3>",
           f"<p><b>Câu hỏi:</b> {html.escape(it['question'])}<br><b>Evidence:</b> {html.escape(it.get('evidence') or '')}<br>"
           f"<b>Vì sao chọn câu này:</b> {html.escape(GHI_CHU[it['uid']][1])}</p>", sql_dap_an(it["SQL"])]
    for k in cau_hinh:
        rec, log = kq[(it["uid"], k)]["rec"], kq[(it["uid"], k)]["log"]
        ok = "✅ đúng" if rec["correct"] else f"❌ sai ({rec['outcome']})"
        out.append(f"<h4>{TEN[k]}: {ok} · {rec['calls']} lần gọi · {rec['prompt_tokens']:,} token vào · "
                   f"{rec['completion_tokens']:,} token ra</h4>")
        if k != "dailsql" and rec.get("objects"):
            out.append("<p style='font-size:13px'>Schema đưa vào: <code>" + html.escape(", ".join(rec["objects"])) + "</code>"
                       + ("<br>Giá trị khớp câu hỏi: <code>" + html.escape(", ".join(f"{v!r} ∈ {t}.{col}" for v, t, col in rec["value_links"][:4])) + "</code>"
                          if rec.get("value_links") else "") + "</p>")
        seen = set()
        for i, x in enumerate(log, 1):
            key = (x["prompt"], json.dumps(x["opts"], sort_keys=True))
            lap = " · <i>prompt trùng lần trước, không tính thêm</i>" if key in seen else ""
            seen.add(key)
            out.append(f"<details><summary>Lần {i}. {html.escape(GIAI_DOAN.get(x['stage'], x['stage']))} · {x['pt']:,} → "
                       f"{x['ct']:,} token{lap}</summary><b>Prompt gửi đi</b>{pre(x['prompt'])}<b>Câu trả lời</b>{pre(x['text'])}</details>")
        out.append("<b>SQL cuối</b>" + pre(rec["final_sql"], 0))
    return HTML("\n".join(out))


def _truc(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", color="#e6e5e0", linewidth=0.8)
    ax.set_axisbelow(True)


def tong_hop_goi_llm(cau, cau_hinh, kq):
    """Bảng token/chi phí và biểu đồ số lần gọi LLM mỗi câu, chia theo giai đoạn."""
    import matplotlib.pyplot as plt
    n = len(cau)
    dem = {k: Counter() for k in cau_hinh}
    bang = ["| Phương pháp | Đúng | Gọi LLM / câu | Token vào / câu | Token ra / câu | USD / câu (giá gpt-4o-mini) |",
            "|---|---|---|---|---|---|"]
    for k in cau_hinh:
        for it in cau:
            seen = set()
            for x in kq[(it["uid"], k)]["log"]:
                key = (x["prompt"], json.dumps(x["opts"], sort_keys=True))
                if key not in seen:               # cùng cách đếm với chương trình chính: prompt trùng không tính
                    seen.add(key)
                    dem[k][NHOM.get(x["stage"], "Giai đoạn 1 / vòng 1")] += 1
        recs = [kq[(it["uid"], k)]["rec"] for it in cau]
        tin, tout = sum(r["prompt_tokens"] for r in recs) / n, sum(r["completion_tokens"] for r in recs) / n
        bang.append(f"| {TEN[k]} | {sum(r['correct'] for r in recs)}/{n} | {sum(r['calls'] for r in recs) / n:.2f} | "
                    f"{tin:,.0f} | {tout:,.0f} | {tin * 0.15e-6 + tout * 0.6e-6:.5f} |")
    display(Markdown("\n".join(bang)))
    fig, ax = plt.subplots(figsize=(8, 0.6 * len(cau_hinh) + 1.2))
    ys = list(range(len(cau_hinh)))[::-1]
    left = [0.0] * len(cau_hinh)
    for g in THU_TU:
        w = [dem[k][g] / n for k in cau_hinh]
        if any(w):
            ax.barh(ys, w, left=left, height=0.5, color=MAU[g], edgecolor="white", linewidth=2, label=g)
            left = [a + b for a, b in zip(left, w)]
    for y, t in zip(ys, left):
        ax.text(t + 0.05, y, f"{t:.2f}", va="center", fontsize=9, color="#52514e")
    ax.set_yticks(ys, [TEN[k] for k in cau_hinh])
    ax.set_xlabel(f"Số lần gọi LLM trung bình mỗi câu ({n} câu demo)")
    ax.set_xlim(0, max(left) + 0.6)
    _truc(ax)
    ax.legend(frameon=False, fontsize=8, ncol=4, loc="lower left", bbox_to_anchor=(0, 1.0))
    plt.tight_layout()
    plt.show()


# Kết quả lần chạy đủ 500 câu (results_new/cmp_gpt-4o-mini_d2025)
KQ500 = {   # EX (%) dễ / trung bình / khó / tổng, % ảo giác schema, gọi LLM/câu, token vào/câu, USD/500 câu
    "dailsql":  dict(d=66.2, tb=47.6, k=32.4, ex=50.0, ao=9.2,  goi=1.75, tok=2946,  usd=0.36),
    "vsql_all": dict(d=62.8, tb=43.2, k=28.4, ex=46.0, ao=12.6, goi=2.00, tok=14412, usd=1.12),
    "vsql_llm": dict(d=62.2, tb=46.4, k=25.5, ex=46.8, ao=10.4, goi=3.00, tok=8219,  usd=0.66),
    "vsqlplus": dict(d=68.2, tb=49.6, k=37.3, ex=52.6, ao=7.0,  goi=1.93, tok=5984,  usd=0.48),
}
# giai đoạn 1 (SQL nháp trước giai đoạn 2), số câu trên 500: đúng / lỗi thực thi / chạy được nhưng sai
GD1 = {"dailsql": (248, 70, 182), "vsql_all": (226, 85, 189), "vsql_llm": (216, 75, 209), "vsqlplus": (267, 20, 213)}


def ket_qua_500():
    import matplotlib.pyplot as plt
    K = list(KQ500)
    display(Markdown(
        "| Phương pháp | Dễ (148) | TB (250) | Khó (102) | **Tổng** | So với DAIL-SQL | Ảo giác schema | Gọi LLM/câu | Token vào/câu | USD/500 câu |\n"
        "|---|---|---|---|---|---|---|---|---|---|\n"
        + "\n".join(f"| {TEN[k]} | {x['d']} | {x['tb']} | {x['k']} | **{x['ex']}** | "
                    + ("—" if k == "dailsql" else f"{x['ex'] - 50.0:+.1f} điểm")
                    + f" | {x['ao']}% | {x['goi']} | {x['tok']:,} | {x['usd']} |" for k, x in KQ500.items())
        + "\n\nV-SQL (tái hiện) thấp hơn DAIL-SQL 3–4 điểm; V-SQL+ cao hơn DAIL-SQL 2,6 điểm và cao hơn V-SQL (LLM chọn view) 5,8 điểm."))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw=dict(width_ratios=[1, 1.4]))
    ys = list(range(len(K)))[::-1]
    ex = [KQ500[k]["ex"] for k in K]
    a1.barh(ys, ex, height=0.45, color="#2a78d6")
    for y, v in zip(ys, ex):
        a1.text(v + 0.8, y, f"{v:.1f}", va="center", fontsize=9, color="#52514e")
    a1.set_yticks(ys, [TEN[k] for k in K])
    a1.set_xlim(0, 70)
    a1.set_xlabel("EX (%) trên 500 câu")
    nhan = ["Đúng", "Lỗi thực thi (sai tên cột, hàm, cú pháp)", "Chạy được nhưng sai kết quả"]
    left = [0] * len(K)
    for j, (nh, m) in enumerate(zip(nhan, ["#2a78d6", "#eb6834", "#1baf7a"])):
        w = [GD1[k][j] for k in K]
        a2.barh(ys, w, left=left, height=0.45, color=m, edgecolor="white", linewidth=2, label=nh)
        for y, l0, v in zip(ys, left, w):
            a2.text(l0 + v / 2, y, str(v), va="center", ha="center", fontsize=8, color="white")
        left = [a + b for a, b in zip(left, w)]
    a2.set_yticks(ys, [""] * len(K))
    a2.set_xlim(0, 500)
    a2.set_xlabel("Giai đoạn 1: số câu trên 500")
    a2.legend(frameon=False, fontsize=8, ncol=3, loc="lower left", bbox_to_anchor=(0, 1.0))
    for a in (a1, a2):
        _truc(a)
    plt.tight_layout()
    plt.show()


print("Đã nạp hàm hiển thị.")

---
# Demo (khoảng 3 phút; mỗi ô chạy khoảng 1 giây, chạy lại thoải mái)

## Phần 1. Tái hiện V-SQL

**Ý tưởng của paper (Hình 1):** CSDL có nhiều khoá ngoại làm LLM dễ JOIN sai và đoán sai tên cột. V-SQL gộp các bảng liên quan thành
**view** (bảng ảo), rồi sinh SQL theo 2 giai đoạn:
1. **Giai đoạn 1 (Hình 4):** LLM viết *dummy SQL* trên view; vì view đã gộp sẵn cột của bảng liên quan nên ít JOIN.
2. **Giai đoạn 2 (Hình 5):** LLM nhận định nghĩa view và viết lại dummy SQL thành SQL trên bảng gốc.

**Cách tái hiện:**
- **View:** paper (mục 6) tự thiết kế view bằng tay dựa trên các phép JOIN trong SQL đáp án, nhưng không công bố. Chúng tôi dựng tự
  động theo cùng nguyên tắc: mỗi nhóm bảng được JOIN cùng nhau trong đáp án cho một view `v_<bảng gốc>`, viết đúng kiểu Hình 3 (cột id
  của bảng tra cứu như `colour`, `gender` được thay bằng giá trị). Mỗi view có đúng một dòng cho mỗi dòng của bảng gốc (đã kiểm tra).
- **D′:** mọi view + các bảng không bị view thay thế. Bảng tra cứu đã gộp vào view thì biến mất (Hình 1(A)).
- **Prompt:** Hình 4 và Hình 5 giữ nguyên từng chữ. Giai đoạn 2 luôn chạy; chỉ đưa các bảng gốc mà dummy SQL thật sự dùng (mục 4.5).
- **Chọn view:** paper chỉ nói schema "related to the query, retrieved", không nói cách chọn. Có 2 bản: *đưa hết view*, và *LLM chọn
  view* (thêm 1 lần gọi LLM, phần tự thêm).

In [ ]:
#@title 1. View của V-SQL trên CSDL superhero, so với Hình 3 của paper
from vsql.methods import vsql_dprime, P_VSQL_1, P_VSQL_2
c = CTX["superhero"]
views, tabs = vsql_dprime(c)
display(Markdown(f"**CSDL superhero:** {len(c.schema.tables)} bảng (`" + "`, `".join(c.schema.tables) + "`)  \n"
                 f"**D′ của V-SQL:** {len(views)} view (`" + "`, `".join(v.name for v in views) + "`)"
                 + (f" + {len(tabs)} bảng (`" + "`, `".join(tabs) + "`)" if tabs else " , không còn bảng nào") + "."))
v = next(v for v in views if v.name == "v_superhero")
HINH3 = ["id", "superhero_name", "full_name", "gender", "eye_colour", "hair_colour", "skin_colour", "race", "publisher_name",
         "alignment", "height_cm", "weight_kg"]
cot = [x.name for x in v.cols]
display(Markdown(f"**`v_superhero` do chương trình dựng** — {'trùng đủ 12 cột, đúng thứ tự' if cot == HINH3 else 'khác Hình 3: ' + str(cot)} "
                 "của Hình 3 trong paper:"))
display(HTML(pre(v.ddl())))

STATS = [("california_schools", 3, 2, 1), ("card_games", 6, 4, 1), ("codebase_community", 8, 7, 7),
         ("debit_card_specializing", 5, 2, 2), ("european_football_2", 7, 3, 3), ("financial", 8, 7, 7), ("formula_1", 13, 8, 8),
         ("student_club", 8, 5, 5), ("superhero", 10, 3, 3), ("thrombosis_prediction", 3, 2, 2), ("toxicology", 4, 3, 3)]
display(Markdown("**Số view trên 11 CSDL** (mọi view đều giữ đúng số dòng của bảng gốc):\n\n| CSDL | Bảng | View V-SQL (theo JOIN đáp án) | View V-SQL+ (tự động) |\n|---|---|---|---|\n"
                 + "\n".join(f"| {a} | {b} | {x} | {y} |" for a, b, x, y in STATS)
                 + f"\n| **Tổng** | {sum(s[1] for s in STATS)} | {sum(s[2] for s in STATS)} | {sum(s[3] for s in STATS)} |"))
display(HTML("<details><summary><b>Prompt Hình 4 (giai đoạn 1), nguyên văn</b></summary>" + pre(P_VSQL_1) + "</details>"
             "<details><summary><b>Prompt Hình 5 (giai đoạn 2), nguyên văn</b></summary>" + pre(P_VSQL_2) + "</details>"))

## Phần 2. Cải tiến V-SQL+

V-SQL+ giữ khung 2 giai đoạn và prompt Hình 5 của paper, thay đổi những chỗ sau:

| Thành phần | V-SQL (tái hiện) | V-SQL+ |
|---|---|---|
| **Hướng 1: tạo view** | Theo các phép JOIN trong SQL đáp án (nhìn đáp án của cả bộ dữ liệu) | **Tự động theo khoá ngoại**, không nhìn đáp án; giữ cả cột id lẫn cột giá trị; bảng tra cứu vẫn hiện |
| **Hướng 2: chọn view** | Đưa hết, hoặc LLM chọn | **Truy hồi** BM25 theo cột và theo view, cộng điểm cho view chứa giá trị khớp câu hỏi; không gọi LLM |
| Prompt giai đoạn 1 | Hình 4 | **Prompt mới**: ghi rõ SQLite, giải thích view, schema `CREATE VIEW` có chú thích nguồn gốc cột |
| Liên kết giá trị | Không | **Có**: giá trị trong CSDL khớp câu hỏi được đưa vào prompt (đúng chính tả, hoa thường) |
| Sửa lỗi | Không | **Có**: SQL lỗi tên cột/bảng thì gửi lại kèm chỗ cột thật sự nằm, tối đa 2 lần |
| Giai đoạn 2 | Hình 5, luôn chạy | Hình 5, chỉ khi dummy SQL dùng view |

**Đóng góp của từng thành phần** (500 câu, gpt-4o-mini; mỗi dòng so bản V-SQL+ đầy đủ với bản bỏ đúng thành phần đó):

| Thành phần | Thay đổi EX |
|---|---|
| Prompt giai đoạn 1 mới (so với dùng Hình 4 + schema Bảng 6) | **+8,0 điểm** |
| Liên kết giá trị | +4,2 điểm |
| Sửa lỗi tên cột/bảng | +0,6 điểm |
| Bầu chọn nhiều mẫu (tốn gấp khoảng 3 lần token) | −0,2 điểm, nên demo dùng bản bỏ bầu chọn |
| Cả hướng 2 (chọn view + liên kết giá trị + sửa lỗi + bầu chọn), so với đưa hết view | +4,4 điểm |

In [ ]:
#@title 2. View tự động của V-SQL+ và liên kết giá trị (CSDL superhero)
from vsql.autoview import describe
from vsql.linking import link_values
from vsql.methods import P_PLUS
c = CTX["superhero"]
va = next(v for v in c.auto if v.name == "v_superhero")
display(Markdown("**`v_superhero` tự động của V-SQL+** — dựng từ khoá ngoại, giữ cả `eye_colour_id` lẫn `eye_colour`; mỗi cột có "
                 "chú thích nguồn gốc, mô tả và giá trị mẫu:"))
display(HTML(pre(describe(va, c.schema, c.desc, ARGS.samples))))
it = data[266]
links = link_values(c.vindex, it["question"], it.get("evidence"))
display(Markdown(f"**Liên kết giá trị, ví dụ uid 266:** \"{it['question']}\"  \nEvidence: \"{it['evidence']}\"  \n"
                 "Giá trị thật trong CSDL khớp với câu hỏi (đưa vào prompt ở mục `# matched database values`):\n\n"
                 + "\n".join(f"- `{v!r}` là giá trị của cột `{t}.{col}`" for v, t, col, _ in links[:5])))
display(HTML("<details><summary><b>Prompt giai đoạn 1 của V-SQL+</b></summary>" + pre(P_PLUS) + "</details>"))

## Phần 3. Đánh giá trên các câu demo

Chạy các phương pháp đã chọn trên các câu demo, chấm bằng cách chạy SQL dự đoán và SQL đáp án trên CSDL, so sánh tập kết quả.
Đổi số câu hoặc phương pháp rồi chạy lại ô 3 → 5, không cần chạy lại phần Chuẩn bị.

In [ ]:
#@title 3. Chạy các phương pháp trên câu demo
#@markdown **Số câu** (1–10): từ 3 câu trở lên có đủ dễ / trung bình / khó; 10 câu = 3 / 5 / 2 như bộ dữ liệu.
SO_CAU = 3  #@param {type:"slider", min:1, max:10, step:1}
#@markdown **Phương pháp**
DAIL_SQL = True  #@param {type:"boolean"}
VSQL_DUA_HET_VIEW = True  #@param {type:"boolean"}
VSQL_LLM_CHON_VIEW = True  #@param {type:"boolean"}
VSQL_PLUS = True  #@param {type:"boolean"}

CAU_HINH = [k for k, on in zip(TAT_CA, [DAIL_SQL, VSQL_DUA_HET_VIEW, VSQL_LLM_CHON_VIEW, VSQL_PLUS]) if on]
assert CAU_HINH, "Chọn ít nhất một phương pháp"
CAU = [data[u] for u, _, _ in DS_CAU[:max(1, min(10, int(SO_CAU)))]]
t = time.time()
KQ = {(it["uid"], k): chay(it, k) for it in CAU for k in CAU_HINH}
goi_that = sum(not x["tu_cache"] for r in KQ.values() for x in r["log"])
display(bang_ket_qua(CAU, CAU_HINH, KQ))
print(f"{len(CAU)} câu × {len(CAU_HINH)} phương pháp: {time.time() - t:.1f} giây; {goi_that} lần gọi API thật, còn lại lấy từ cache.")

In [ ]:
#@title 4. Luồng gọi LLM của một câu (bấm vào từng lần gọi để xem prompt và câu trả lời)
XEM_UID = 259  #@param [259, 330, 261, 380, 427, 266, 421, 265, 50, 85]
it = data[int(XEM_UID)]
for k in CAU_HINH:
    if (it["uid"], k) not in KQ:
        KQ[(it["uid"], k)] = chay(it, k)
display(chi_tiet(it, CAU_HINH, KQ))

In [ ]:
#@title 5. Số lần gọi LLM, token, chi phí trên các câu demo
tong_hop_goi_llm(CAU, CAU_HINH, KQ)

## Phần 4. Kết quả trên đủ 500 câu (gpt-4o-mini)

Vài câu demo không đủ để kết luận. Số liệu dưới đây lấy từ lần chạy đủ 500 câu (`results_new/cmp_gpt-4o-mini_d2025`), mọi phương pháp
cùng model, nhiệt độ 0.

In [ ]:
#@title 6. Độ chính xác và phân tích lỗi giai đoạn 1 (500 câu)
ket_qua_500()

## Kết luận

- **Tái hiện V-SQL:** view dựng lại khớp Hình 3; prompt Hình 4 và Hình 5 giữ nguyên. Ở các câu nhiều khoá ngoại, view giúp đúng như paper
  nói (câu demo 259, 380, 421, 427). Nhưng trên đủ 500 câu, V-SQL đạt 46,0–46,8%, thấp hơn DAIL-SQL (50,0%) khoảng 3–4 điểm. Paper báo
  V-SQL 56,0% với GPT-4. Chênh lệch có thể do view của tác giả được thiết kế bằng tay (không công bố), do model khác, và do prompt Hình 4
  zero-shot, không nhắc SQLite.
- **Cải tiến V-SQL+:** 52,6%, cao nhất, hơn V-SQL 5,8 điểm và hơn DAIL-SQL 2,6 điểm, chi phí chỉ hơn DAIL-SQL một chút. Phần lớn mức
  tăng đến từ prompt mới (+8,0 điểm) và liên kết giá trị (+4,2 điểm). Ở giai đoạn 1, V-SQL+ gần như hết lỗi thực thi (20 câu so với 70
  của DAIL-SQL).
- **Điểm còn yếu:** V-SQL+ vẫn có nhiều câu "chạy được nhưng sai" hơn DAIL-SQL, vì DAIL-SQL có 7 ví dụ few-shot. Giai đoạn 2 (Hình 5)
  đôi khi làm hỏng câu đã đúng (câu demo 265). Hướng tiếp theo: thêm ví dụ few-shot cho V-SQL+.